# BarGPT real shard sample audit

Loads one deterministic random block from a certified workstation shard and displays the exact model-ready inputs, causal context indices, autoregressive targets, physical-horizon targets, and optional checkpoint predictions. No ClickHouse writes or shard changes occur.

In [ ]:
from pathlib import Path
import random, sys
import polars as pl
from IPython.display import display

REPO_CANDIDATES = (Path(r'D:/TradingML/codes/quant-research-workbench'), Path(r'D:/TradingCodes/quant-research-workbench'))
REPO = next((path for path in REPO_CANDIDATES if path.exists()), None)
if REPO is None: raise RuntimeError(f'BarGPT repository not found: {REPO_CANDIDATES}')
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
SHARD_CANDIDATES = (
    Path(r'D:/TradingML/runtimes/bar_gpt/v1/offline_shards_v5_pilot'),
    Path(r'D:/TradingML/runtimes/bar_gpt/v1/offline_shards_v5'),
)
SHARD_ROOT = next((path for path in SHARD_CANDIDATES if any(path.glob('tickers/*/*/*.json'))), None)
if SHARD_ROOT is None: raise RuntimeError(f'No populated BarGPT shard catalog found: {SHARD_CANDIDATES}')
CHECKPOINT = Path(r'')  # Optional checkpoint_latest.pt for prediction columns
SEED = 17
TICKERS = ()  # Example: ('AAPL',)
TAIL_CONTEXT_ROWS = 8
print('repository:', REPO)
print('shards:', SHARD_ROOT)

In [ ]:
from research.bar_gpt.v1.shard_data_audit import (
    context_rows, data_config_for_sample, load_audit_sample, sample_manifest, select_random_audit_blocks,
    selected_autoregressive_targets, selected_input_rows, selected_predictions, selected_targets,
)

ref = select_random_audit_blocks(SHARD_ROOT, max_shards=1, samples_per_shard=1, seed=SEED, tickers=TICKERS)[0]
sample = load_audit_sample(SHARD_ROOT, ref)
origin_offset = random.Random(f'{SEED}|{ref.unit_key}|{ref.block_offset}').randrange(sample.block.origin_indices.numel())
manifest = sample_manifest(sample, origin_offset=origin_offset)
display(pl.DataFrame([manifest['reference']]))
display(pl.DataFrame(manifest['context']))
print('selected origin offset:', origin_offset, 'timestamp_us:', manifest['origin_timestamp_us'])

## Exact model-ready input rows visible at the selected origin

In [ ]:
input_rows = selected_input_rows(sample, origin_offset=origin_offset, tail_rows=TAIL_CONTEXT_ROWS)
for view, rows in input_rows.items():
    print(f'\n{view}: last {len(rows)} causally visible rows')
    if not rows:
        print('unavailable/masked')
        continue
    frame = pl.DataFrame(rows)
    display(frame)
    asof = rows[-1]
    metadata = {'row_index', 'bar_start_us', 'bar_end_us', 'available_at_us', 'is_asof'}
    display(pl.DataFrame({'feature': [name for name in asof if name not in metadata], 'value': [asof[name] for name in asof if name not in metadata]}))

## Stored targets and masks

In [ ]:
data_config = data_config_for_sample(sample)
horizon_rows = selected_targets(sample, origin_offset, data_config)
ar_rows = selected_autoregressive_targets(sample, origin_offset)
display(pl.DataFrame(horizon_rows))
display(pl.DataFrame(ar_rows))

## Optional checkpoint outputs versus targets

In [ ]:
if CHECKPOINT and CHECKPOINT.is_file():
    predictions = selected_predictions(sample, origin_offset=origin_offset, checkpoint_path=CHECKPOINT)
    comparison = pl.DataFrame(horizon_rows).join(pl.DataFrame(predictions), on='horizon', how='left')
    display(comparison)
else:
    print('Set CHECKPOINT to a checkpoint_latest.pt to display model predictions beside targets.')